In [0]:
##### Grocery data profiling

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

In [0]:
print("\n[1/8] Loading table...")

TABLE_NAME = "workspace.default.grocery_details_02_12"

try:
    df = spark.table(TABLE_NAME)
    print(f"✓ Table loaded: {TABLE_NAME}")
except:
    try:
        TABLE_NAME = "default.grocery_details_02_12"
        df = spark.table(TABLE_NAME)
        print(f"✓ Table loaded: {TABLE_NAME}")
    except Exception as e:
        print(f"✗ Error: {e}")
        raise

print("\n--- First 10 Rows ---")
df.show(10, truncate=False)


[1/8] Loading table...
✓ Table loaded: workspace.default.grocery_details_02_12

--- First 10 Rows ---
+--------+----------------------------+---------+----------------------------+---------------+----+--------+---------------------------------------------------------------------------+----+--------------------------------------------------------------------------------+
|objectid|coname                      |locnum   |staddr                      |stcity         |zip |naics   |store_type                                                                 |year|shape                                                                           |
+--------+----------------------------+---------+----------------------------+---------------+----+--------+---------------------------------------------------------------------------+----+--------------------------------------------------------------------------------+
|1       |Lamberts Rainbow Fruit      |000479394|777 William T Morrissey Blvd|Dorche

In [0]:
#Basic Info
print("\n" + "="*70)
print("📋 DATASET OVERVIEW")
print("="*70)
 
row_count = df.count()
col_count = len(df.columns)
 
print(f"Total Rows: {row_count:,}")
print(f"Total Columns: {col_count}")
 
print("\n--- Schema ---")
df.printSchema()
 
print("\n--- Column Names ---")
for i, col_name in enumerate(df.columns, 1):
    print(f"{i:2}. {col_name}")


📋 DATASET OVERVIEW
Total Rows: 15,335
Total Columns: 10

--- Schema ---
root
 |-- objectid: long (nullable = true)
 |-- coname: string (nullable = true)
 |-- locnum: string (nullable = true)
 |-- staddr: string (nullable = true)
 |-- stcity: string (nullable = true)
 |-- zip: long (nullable = true)
 |-- naics: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- year: long (nullable = true)
 |-- shape: string (nullable = true)


--- Column Names ---
 1. objectid
 2. coname
 3. locnum
 4. staddr
 5. stcity
 6. zip
 7. naics
 8. store_type
 9. year
10. shape


In [0]:
# Column Analysis
print("\n" + "="*70)
print("📊 DETAILED COLUMN ANALYSIS")
print("="*70)
 
summary_data = []
 
for col_name in df.columns:
    print(f"\n--- {col_name} ---")
 
    col_type = str(df.schema[col_name].dataType)
    print(f"Type: {col_type}")
 
    null_count = df.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / row_count * 100) if row_count > 0 else 0.0
    print(f"Nulls: {null_count:,} ({null_pct:.2f}%)")
 
    distinct_count = df.select(F.col(col_name)).distinct().count()
    print(f"Distinct: {distinct_count:,}")
 
    try:
        samples = (
            df.select(F.col(col_name))
              .where(F.col(col_name).isNotNull())
              .limit(3)
              .collect()
        )
        sample_values = [str(r[0])[:80] for r in samples]
        print(f"Samples: {sample_values}")
    except Exception:
        print("Samples: (could not retrieve)")
 
    summary_data.append({
        "Column": col_name,
        "Type": col_type.replace("Type", ""),
        "Nulls": int(null_count),
        "Null_Pct": round(float(null_pct), 2),
        "Distinct": int(distinct_count)
    })


📊 DETAILED COLUMN ANALYSIS

--- objectid ---
Type: LongType()
Nulls: 0 (0.00%)
Distinct: 15,335
Samples: ['1', '2', '3']

--- coname ---
Type: StringType()
Nulls: 0 (0.00%)
Distinct: 7,892
Samples: ['Lamberts Rainbow Fruit', "Gaspar's Linguica Co Inc", 'Crescent Ridge']

--- locnum ---
Type: StringType()
Nulls: 0 (0.00%)
Distinct: 5,762
Samples: ['000479394', '000925115', '001466424']

--- staddr ---
Type: StringType()
Nulls: 0 (0.00%)
Distinct: 10,107
Samples: ['777 William T Morrissey Blvd', '384 Faunce Corner Rd', '355 Bay Rd']

--- stcity ---
Type: StringType()
Nulls: 0 (0.00%)
Distinct: 469
Samples: ['Dorchester', 'North Dartmouth', 'Sharon']

--- zip ---
Type: LongType()
Nulls: 0 (0.00%)
Distinct: 526
Samples: ['2122', '2747', '2067']

--- naics ---
Type: StringType()
Nulls: 0 (0.00%)
Distinct: 43
Samples: ['44523003', '44521010', '44529907']

--- store_type ---
Type: StringType()
Nulls: 0 (0.00%)
Distinct: 18
Samples: ['Fruit and Vegetable Markets', 'Meat Markets, Fish and Seaf

In [0]:
# Column summary table
print("\n" + "="*70)
print("📋 COLUMN SUMMARY TABLE")
print("="*70)
 
summary_df = pd.DataFrame(summary_data)
try:
    display(summary_df)  # Databricks display
except Exception:
    print(summary_df.to_string(index=False))


📋 COLUMN SUMMARY TABLE


Column,Type,Nulls,Null_Pct,Distinct
objectid,Long(),0,0.0,15335
coname,String(),0,0.0,7892
locnum,String(),0,0.0,5762
staddr,String(),0,0.0,10107
stcity,String(),0,0.0,469
zip,Long(),0,0.0,526
naics,String(),0,0.0,43
store_type,String(),0,0.0,18
year,Long(),0,0.0,2
shape,String(),0,0.0,14415


In [0]:
# Numeric Statistics
print("\n" + "="*70)
print("NUMERIC COLUMN STATISTICS")
print("="*70)
 
from pyspark.sql.types import *
 
numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, IntegerType, LongType, FloatType, DecimalType, ShortType))
]

print(f"Numeric Columns Found: {len(numeric_cols)}")
if numeric_cols:
    print("Columns:", ", ".join(numeric_cols))
    df.select(numeric_cols).describe().show(truncate=False)
else:
    print("No numeric columns found.")


🔢 NUMERIC COLUMN STATISTICS
Numeric Columns Found: 3
Columns: objectid, zip, year
+-------+-----------------+------------------+-----------------+
|summary|objectid         |zip               |year             |
+-------+-----------------+------------------+-----------------+
|count  |15335            |15335             |15335            |
|mean   |7668.0           |1980.7138571894359|2018.502706227584|
|stddev |4426.977524225755|476.06837381197397|1.937251567171538|
|min    |1                |0                 |2017             |
|max    |15335            |2791              |2021             |
+-------+-----------------+------------------+-----------------+



In [0]:
# Missing data
print("\n" + "="*70)
print("MISSING DATA REPORT")
print("="*70)
 
missing_sorted = sorted(summary_data, key=lambda x: x["Null_Pct"], reverse=True)
 
print(f"\n{'Column':<35} {'Missing':<15} {'Percent':<10}")
print("-" * 65)
 
cols_with_missing = 0
for info in missing_sorted:
    if info["Null_Pct"] > 0:
        cols_with_missing += 1
        print(f"{info['Column']:<35} {info['Nulls']:<15,} {info['Null_Pct']:<10.2f}")
 
print(f"\n✓ {cols_with_missing} columns have missing data")


MISSING DATA REPORT

Column                              Missing         Percent   
-----------------------------------------------------------------

✓ 0 columns have missing data


In [0]:
# Duplicate Analysis
print("\n" + "="*70)
print("DUPLICATE ANALYSIS")
print("="*70)
 
distinct_rows = df.distinct().count()
duplicate_rows = row_count - distinct_rows
duplicate_pct = (duplicate_rows / row_count * 100) if row_count > 0 else 0.0
 
print(f"Total Rows:     {row_count:>12,}")
print(f"Distinct Rows:  {distinct_rows:>12,}")
print(f"Duplicates:     {duplicate_rows:>12,} ({duplicate_pct:.2f}%)")


DUPLICATE ANALYSIS
Total Rows:           15,335
Distinct Rows:        15,335
Duplicates:                0 (0.00%)


In [0]:
# Categorical frequencies
print("\n" + "="*70)
print("CATEGORICAL COLUMN FREQUENCIES")
print("="*70)
 
categorical_cols = []
for col_name in df.columns:
    if col_name not in numeric_cols:
        try:
            dcount = df.select(F.col(col_name)).distinct().count()
            if 1 < dcount <= 100:
                categorical_cols.append((col_name, dcount))
        except Exception:
            pass
 
print(f"Found {len(categorical_cols)} categorical columns (2–100 unique values).")
for col_name, dcount in categorical_cols[:3]:
    print(f"\n--- {col_name} (distinct: {dcount}) ---")
    df.groupBy(F.col(col_name)).count().orderBy(F.desc("count")).show(10, truncate=False)


CATEGORICAL COLUMN FREQUENCIES
Found 2 categorical columns (2–100 unique values).

--- naics (distinct: 43) ---
+--------+-----+
|naics   |count|
+--------+-----+
|        |9574 |
|44512001|2011 |
|44511003|1687 |
|44611009|723  |
|45231912|368  |
|44522004|202  |
|44523003|194  |
|44521006|159  |
|45221001|98   |
|44529912|73   |
+--------+-----+
only showing top 10 rows

--- store_type (distinct: 18) ---
+---------------------------------------------------------------------------+-----+
|store_type                                                                 |count|
+---------------------------------------------------------------------------+-----+
|Supermarkets/Other Grocery (Exc Convenience) Strs                          |3592 |
|Convenience Stores                                                         |3176 |
|Convenience Stores, Pharmacies, and Drug Stores                            |2792 |
|Supermarket or Other Grocery                                               |1815 |
|

In [0]:
# Final Summary
print("\n" + "="*70)
print("PROFILING COMPLETE!")
print("="*70)
 
total_cells = row_count * col_count
null_cells = sum(x["Nulls"] for x in summary_data)
complete_cells = total_cells - null_cells
completeness = (complete_cells / total_cells * 100) if total_cells > 0 else 0.0
 
print(f"\nOVERALL SUMMARY:")
print(f"  • Total Rows:              {row_count:>12,}")
print(f"  • Total Columns:           {col_count:>12,}")
print(f"  • Total Data Points:       {total_cells:>12,}")
print(f"  • Complete Data Points:    {complete_cells:>12,}")
print(f"  • Missing Data Points:     {null_cells:>12,}")
print(f"  • Data Completeness:       {completeness:>11.2f}%")
 
print(f"\nCOLUMN BREAKDOWN:")
print(f"  • Numeric Columns:         {len(numeric_cols):>12}")
print(f"  • Categorical Columns:     {len(categorical_cols):>12}")
print(f"  • Columns with Missing:    {cols_with_missing:>12}")
 
print(f"\nDATA QUALITY:")
print(f"  • Duplicate Rows:          {duplicate_rows:>12,} ({duplicate_pct:.2f}%)")
print(f"  • Unique Rows:             {distinct_rows:>12,}")


PROFILING COMPLETE!

OVERALL SUMMARY:
  • Total Rows:                    15,335
  • Total Columns:                     10
  • Total Data Points:            153,350
  • Complete Data Points:         153,350
  • Missing Data Points:                0
  • Data Completeness:            100.00%

COLUMN BREAKDOWN:
  • Numeric Columns:                    3
  • Categorical Columns:                2
  • Columns with Missing:               0

DATA QUALITY:
  • Duplicate Rows:                     0 (0.00%)
  • Unique Rows:                   15,335
